# Objective

Illustrate fine-tuning Mistral to follow a specific output format (for e.g., JSON).

# Setup

In [ ]:
# #@title Run this cell to setup Unsloth on Colab
# !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install -q --no-deps xformers==0.0.28.post1 trl==0.11.1 peft==0.13.0 accelerate==0.34.2 bitsandbytes==0.44.1
# !pip install triton==3.0.0

In [1]:
!pip install "trl<0.15.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.9/313.9 kB 7.6 MB/s eta 0:00:00


Run this cell to setup Unsloth on Colab

In [2]:
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 3.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.2/379.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 852.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2

In [3]:
#download datasets evaluate rouge_score and bert score
!pip install -q datasets==3.0.1 evaluate==0.4.3 bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 12.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.1.1 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 3.0.1 which is incompatible.
torchaudio 2.9.0+cu126 requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [4]:
import torch
import json

from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Model

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True
)

==((====))==  Unsloth 2026.1.1: Fast Mistral patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
tokenizer.add_bos_token, tokenizer.add_eos_token

(True, False)

In [7]:
EOS_TOKEN = tokenizer.eos_token

# Data

In [8]:
dataset = load_dataset("pgurazada1/entities-laptop")

README.md:   0%|          | 0.00/154 [00:00<?, ?B/s]

entities-train.json: 0.00B [00:00, ?B/s]

entities-validation.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/39 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/20 [00:00<?, ? examples/s]

In [9]:
training_dataset = dataset['train']
validation_dataset = dataset['validation']

In [10]:
training_dataset[0]

{'entities': {'battery_life': 'impressive, lasting a solid 10 hours',
  'design': 'sleek',
  'keyboard': 'backlit',
  'performance': 'handles multitasking effortlessly',
  'sturdiness': 'aluminum chassis',
  'trackpad': 'responsive'},
 'review': "The laptop's battery life is impressive, lasting a solid 10 hours on a single charge. The sleek design and backlit keyboard enhance the overall experience. Performance-wise, it handles multitasking effortlessly. The aluminum chassis adds sturdiness, and the trackpad is responsive."}

In [11]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [12]:
def prompt_formatter(example, prompt_template):
    instruction="Extract entities in the input review in a JSON format."
    review=example["review"]
    entities=example["entities"]

    formatted_prompt = prompt_template.format(instruction, review, entities) + EOS_TOKEN

    return {'formatted_prompt': formatted_prompt}

In [13]:
formatted_training_dataset = training_dataset.map(
    prompt_formatter,
    fn_kwargs={'prompt_template': alpaca_prompt}
)

Map:   0%|          | 0/39 [00:00<?, ? examples/s]

In [14]:
formatted_validation_dataset = validation_dataset.map(
    prompt_formatter,
    fn_kwargs={'prompt_template': alpaca_prompt}
)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [15]:
formatted_training_dataset[0]

{'entities': {'battery_life': 'impressive, lasting a solid 10 hours',
  'design': 'sleek',
  'keyboard': 'backlit',
  'performance': 'handles multitasking effortlessly',
  'sturdiness': 'aluminum chassis',
  'trackpad': 'responsive'},
 'review': "The laptop's battery life is impressive, lasting a solid 10 hours on a single charge. The sleek design and backlit keyboard enhance the overall experience. Performance-wise, it handles multitasking effortlessly. The aluminum chassis adds sturdiness, and the trackpad is responsive.",
 'formatted_prompt': "Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nExtract entities in the input review in a JSON format.\n\n### Input:\nThe laptop's battery life is impressive, lasting a solid 10 hours on a single charge. The sleek design and backlit keyboard enhance the overall experience. Performance-wise, it handles multitasking 

# Fine-tuning

In [21]:
model = FastLanguageModel.get_peft_model(
    model,
    r=4,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing=True,
    random_state=42,
    loftq_config=None
)

Unsloth: Already have LoRA adapters! We shall skip this step.


Notice how $r = \alpha/4$.

In [30]:
def formatting_func(example):
    # example["formatted_prompt"] can be:
    #  - a single string
    #  - a list of strings (in batched mode)

    text = example["formatted_prompt"]

    # if it is already a list -> return as-is
    if isinstance(text, list):
        return text

    # if it is a single string -> wrap inside a list
    return [text]

In [33]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_training_dataset,
    eval_dataset=formatted_validation_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    formatting_func=formatting_func,
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=20,
        eval_strategy="epoch",
        save_strategy='epoch',
        metric_for_best_model="eval_loss",
        load_best_model_at_end=True,
        greater_is_better=False,
        learning_rate=5e-5,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
        output_dir="outputs"
    )
)

In [34]:
training_history = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 39 | Num Epochs = 20 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 10,485,760 of 7,252,217,856 (0.14% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,1.519400,1.243856
2,0.809300,0.710260
3,0.491300,0.366304
4,0.320100,0.237380
5,0.282100,0.188608
6,0.210300,0.167054
7,0.208000,0.153815
8,0.159400,0.142402
9,0.161800,0.137458
10,0.129900,0.126501


Unsloth: Not an error, but MistralForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


# Inference

In [35]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [36]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [37]:
test_review = """
This laptop impresses with its remarkable battery life, lasting well beyond the competition. The sleek design adds a touch of elegance, complemented by a comfortable keyboard that enhances productivity. Performance is stellar, seamlessly handling demanding tasks. Sturdiness is evident in its durable build, ensuring longevity. The trackpad is responsive and precise, contributing to an overall delightful user experience. In summary, this laptop excels across the board, making it a top choice for those seeking a reliable and high-performing device.
"""

In [38]:
instruction = "Extract entities in the input review in a JSON format."

In [39]:
inputs = tokenizer(
[
    alpaca_prompt.format(
        instruction,
        test_review,
        "", # leave output blank for generation
    )
], return_tensors="pt").to("cuda")

In [40]:
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id
)

In [41]:
print(
    tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
        cleanup_tokenization_spaces=True
    )
)

{'battery_life': 'remarkable, lasting well beyond the competition', 'design': 'sleek, adds a touch of elegance', 'keyboard': 'comfortable, enhances productivity', 'performance': 'stellar, seamlessly handling demanding tasks', 'sturdiness': 'durable build, ensures longevity', 'trackpad': 'responsive and precise', 'user_experience': 'overall delightful'}


# Evaluation

In [42]:
examples = [
    "The Dell XPS 13 impresses with its sleek design and remarkable battery life. " \
    "The backlit keyboard offers a comfortable typing experience, while the powerful performance ensures seamless multitasking. " \
    "Sturdiness is top-notch, but the trackpad could be more responsive.",
    "Apple's MacBook Air M2 is a design masterpiece, ultra-thin and lightweight. "
    "The keyboard is a joy to type on, but its standout feature is the incredible battery life. "\
    "Performance-wise, it's a powerhouse, though the sturdiness could be enhanced.",
    "The HP Spectre x360 blends elegance with versatility. "\
    "The keyboard is tactile, and the battery life is commendable. "\
    "Performance-wise, it's a workhorse, but the sturdiness feels slightly compromised. "\
    "The trackpad, however, is responsive and intuitive.",
    "Lenovo's ThinkPad X1 Carbon exudes durability with its robust design. "\
    "The keyboard is superb for long typing sessions. "\
    "Although the battery life falls short of some competitors, the performance is stellar. "\
    "The trackpad is precise, but the design lacks a modern flair.",
    "The Asus ROG Zephyrus G14 is a powerhouse for gaming and productivity. "\
    "While the design is gaming-centric, the keyboard is comfortable, and the performance is exceptional. "\
    "Battery life is average, but the sturdiness is noteworthy. The trackpad could use some improvement."
]

In [43]:
for example in examples:
  inputs = tokenizer(
      [
          alpaca_prompt.format(
              instruction,
              example,
              "", # leave output blank for generation
          )
      ], return_tensors="pt").to("cuda")

  outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    temperature=0,
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id
  )

  output_json = tokenizer.decode(
      outputs[0][inputs.input_ids.shape[-1]:],
      skip_special_tokens=True,
      cleanup_tokenization_spaces=True
  )

  print(example)
  print(output_json.replace("'", '"'))
  print("***\n")

The Dell XPS 13 impresses with its sleek design and remarkable battery life. The backlit keyboard offers a comfortable typing experience, while the powerful performance ensures seamless multitasking. Sturdiness is top-notch, but the trackpad could be more responsive.
{"design": "sleek", "battery_life": "remarkable", "keyboard": "backlit, comfortable", "performance": "seamless multitasking", "sturdiness": "top-notch", "trackpad": "could be more responsive"}
***

Apple's MacBook Air M2 is a design masterpiece, ultra-thin and lightweight. The keyboard is a joy to type on, but its standout feature is the incredible battery life. Performance-wise, it's a powerhouse, though the sturdiness could be enhanced.
{"design": "design masterpiece, ultra-thin and lightweight", "keyboard": "keyboard is a joy to type on", "battery_life": "incredible battery life", "performance": "powerhouse", "sturdiness": "could be enhanced"}
***

The HP Spectre x360 blends elegance with versatility. The keyboard is ta